# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, referencing all data entities by their `@id` as per the Croissant specification.

### Dataset Source
The dataset is defined by a Croissant schema available at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install `mlcroissant` library if not already installed
!pip install --quiet mlcroissant pandas

## 1. Data Loading
Load the metadata and records from the FAIR² dataset using `mlcroissant`. The dataset object exposes the Croissant metadata and allows iteration over record sets via their `@id`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display dataset metadata
print("Dataset Name: ", getattr(metadata, 'name', None))
print("Description: ", getattr(metadata, 'description', None))
print("Identifier: ", getattr(metadata, 'identifier', None))
print("Authors: ", getattr(metadata, 'author', None))

## 2. Data Overview
Review available record sets, their `@id`s, and associated fields. This will help us reference the correct entities for further extraction and processing.
Let's list all the record sets present in the dataset, and show their `@id` and available fields.

In [ ]:
# List all record sets and their fields with @id

if hasattr(metadata, "record_sets") and metadata.record_sets:
    record_sets = metadata.record_sets
else:
    # Try legacy Croissant @id schema
    record_sets = []
    if hasattr(metadata, "recordSet"):
        record_sets = getattr(metadata, "recordSet")
    elif hasattr(metadata, "record_sets"):
        record_sets = getattr(metadata, "record_sets")

if not record_sets or len(record_sets) == 0:
    # Try to list record sets from the dataset instance
    print("No record sets found in metadata.@record_sets. Discovering from dataset...")
    # mlcroissant from v0.13.0 provides `record_set_ids` property for listing all record sets by @id
    try:
        record_set_ids = dataset.record_set_ids
        print("Record sets present in the dataset:")
        for rsid in record_set_ids:
            print(f" - @id: {rsid}")
    except AttributeError:
        print("Dataset does not expose record set ids directly. Please inspect the metadata JSON if needed.")
        record_set_ids = []
else:
    record_set_ids = [getattr(rs, "@id", getattr(rs, "id", None)) for rs in record_sets]
    print("Record sets defined in metadata:")
    for rs, rsid in zip(record_sets, record_set_ids):
        print(f" - @id: {rsid}")
        if hasattr(rs, "fields") and rs.fields:
            print("   Fields:")
            for f in rs.fields:
                print(f"     - @id: {getattr(f, '@id', getattr(f, 'id', None))}")
        elif hasattr(rs, "field") and rs.field:
            print("   Fields:")
            for f in rs.field:
                print(f"     - @id: {getattr(f, '@id', getattr(f, 'id', None))}")

# For programmatic reference, let's store in a list for next steps
if not record_set_ids or len(record_set_ids) == 0:
    # List any via dataset API
    record_set_ids = getattr(dataset, "record_set_ids", [])

## 3. Data Extraction
Load data from a specific record set, referenced by its `@id`, into pandas DataFrames. Record sets and fields must be referenced by their `@id` for robust programmatic access.

*Note: If you do not see any record sets listed in the previous cell, check the Croissant schema for valid record set `@id`s.*

In [ ]:
# Extract records from discovered record sets into pandas DataFrames
dataframes = {}
if not record_set_ids:
    print("No record set @id discovered. Please check the dataset schema.")
else:
    print(f"Extracting data for record sets: {record_set_ids}")
    for record_set_id in record_set_ids:
        print(f"Loading records from @id: {record_set_id}")
        try:
            records = list(dataset.records(record_set=record_set_id))
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"{len(df)} records loaded. Columns: {df.columns.tolist()}")
        except Exception as e:
            print(f"Could not read records for {record_set_id}: {e}")
    # Display the columns of the first record set as an example
    if dataframes:
        sample_rsid = list(dataframes.keys())[0]
        print(f"\nColumns in the first record set @{sample_rsid}:")
        print(dataframes[sample_rsid].columns.tolist())
        display(dataframes[sample_rsid].head())

## 4. Exploratory Data Analysis (EDA)
Process a numeric field: Filter records, normalize values, and group data by categorical attributes. Make sure to use the correct field `@id`s when referencing columns.

*Modify the example below to select meaningful numeric and group fields, referencing them by their `@id` from the previous data overview.*

In [ ]:
# --- EDA Example: Replace these with meaningful field @id from your dataset --- #
# Set your record set and field @id found in the overview step above
selected_record_set_id = None
if len(dataframes) > 0:
    selected_record_set_id = list(dataframes.keys())[0]

# Choose a numeric field and a grouping field (by @id/column name)
# Inspect available columns
if selected_record_set_id:
    df = dataframes[selected_record_set_id]
    print(f"Columns available: {df.columns.tolist()}")

    # Try to suggest a numeric column for demonstration
    # We'll pick the first column that appears numeric
    possible_numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not possible_numeric_fields:
        # Try to convert all columns to numeric, ignore errors
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col], errors='ignore')
            except Exception:
                pass
        possible_numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]

    if possible_numeric_fields:
        numeric_field_id = possible_numeric_fields[0]
        print(f"Using numeric field: {numeric_field_id}")
        threshold = df[numeric_field_id].quantile(0.95) if len(df) > 10 else 10
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalize this numeric field
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, norm_col]].head())

        # Try to select a categorical/group field
        possible_group_fields = [col for col in df.columns if df[col].dtype == 'object']
        if possible_group_fields:
            group_field_id = possible_group_fields[0]
            print(f"Grouping by: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
            print(grouped_df.head())
        else:
            print("No categorical/group field found for grouping.")
    else:
        print("No numeric field detected in this record set.")
else:
    print("No dataframes found for any record set.")

## 5. Visualization
Visualize the distribution of the selected numeric field or the relationship between two features using matplotlib or seaborn.

*Adjust columns as needed for your dataset. Use `@id` for field selections when possible.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Plot histogram of the selected numeric field and boxplot by group
if selected_record_set_id and 'numeric_field_id' in locals() and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if 'group_field_id' in locals() and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("Not enough numeric data for visualization.")

## 6. Conclusion
In this notebook, we loaded and explored a Croissant-formatted dataset with the `mlcroissant` library. We accessed record sets and fields dynamically using their `@id`, illustrated basic EDA, and visualized key features.

**Next steps:** For deeper analysis, refer to the Croissant schema (and its field `@id`s), explore more record sets, or apply advanced statistical or ML analyses tailored to your research question.